## Step 1:- The imports
- pandas
- sentence_transformers (SentenceTransformer)
- chromadb

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer

c:\Users\Vikrant Singh Thakur\OneDrive\Documents\nlpFinal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import chromadb

## Step 2:- 
- You will create a variable which stores the file path for the file which has our chunks.

In [3]:
chunk_dir="../../data/chunked/langChainchunks.csv"

## Step 3:- 
- Load in the csv using the variable name you defined in step 2.

In [4]:
df=pd.read_csv(chunk_dir)

## Step 4:- 
- Load in the embedding model BAAI/bge-base-en-v1.5

In [6]:
model=SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3128.95it/s]


## Step 5:- 
- load in the chunks column from the dataframe which holds the csv -> convert it to string -> convert it all to list.

In [ ]:
chunks=df["chunk_text"].astype(str).tolist()
# Sanity Check :-
# print(len(chunks))
# print(chunks[1793])

1794
with precision, maximizing output. Get started with Mission Control Mission Control 3.0 is designed around minimizing inefficiencies and increasing token output for AI factory operators. By correlating telemetry across domains, orchestrating power intelligently, modularizing the architecture for agility, and enhancing autonomous remediation with AI, it transforms infrastructure from a passive platform into an active participant in performance optimization. Resources: Stay tuned for our latest release notes and implementation guides for NVIDIA Mission Control 3.0. You can also check out the on-demand replay for the NVIDIA GTC 2026 session with Eli Lilly & Company to hear firsthand insights into architecting and deploying high-performance AI infrastructure with powerful, intelligent software.


## Step 6:-
- create the embeddings using the embedding model variable name you created in step 4. (.encode())

In [11]:
embs=model.encode(chunks, normalize_embeddings=True, show_progress_bar=True, batch_size=30)

Batches: 100%|██████████| 60/60 [19:43<00:00, 19.72s/it]


## Step 7:- 
- Convert those embeddings to list and store em in a new variable.
    - `We do this because chromadb expects embeddings as a plain python list of lists and not a numpy array.`

In [12]:
list_embs=embs.tolist()

## Step 8:- 
- Create a saving directory path for the DB.
- use "../../data/chromaDB/chromaDBRaw"

In [13]:
savePath="../../data/chromaDB/chromaDBRaw"

## Step 9:- 
- Create chromaDB persistence client and store it in a variable.
- Syntax:-
    - `chromadb.PersistentClient(filepath)`
# PersistentClient saves the ChromaDB collection to disk automatically.
# Every time you add data or query, it reads and writes from this folder.
# This is different from FAISS where you manually call write_index().
# ChromaDB handles persistence internally — you just point it at a folder which we defined in Step 8.

In [14]:
client=chromadb.PersistentClient(path=savePath)

## Step 10:-
- Create a variable called collection or any name where in you will define its `name` which can be anything and `metadata` as cosine similarity in a list.
- client.getorcreatecollection

In [17]:
collection=client.get_or_create_collection(name="nvidia_chunks", metadata={"hnsw:space":"cosine"})

## Step 11:-
- Create a variable which stores chunk_id from our dataframe --> Converts it to String datatype --> To list.

In [18]:
chunk_ids=df["chunk_id"].astype(str).tolist()

## Step 12:- 
- Create an empty list for storing just the metadata.

In [19]:
metadata=[]

## Step 13:- 
- iterate using a for loop to get `index and all row values` (use df.iterrows())
- And keep storing the metadata in each iteration of rows. (THE ENTIRE ROW MEANING VALUES FOR ALL COLUMNS IN 1 ROW)
- The following is what you want to get:- 
    - ["source"]
    - ["title"]
    - ["url"]
    - ["published"]
- `KEEP IN MIND THAT YOU HAVE TO CONVERT EVERYTHING TO str() as well just to be safe.`

In [21]:
for i, j in df.iterrows():
    metadatas={
        "Source": str(j["source"]),
        "Title": str(j["title"]),
        "url": str(j["url"]),
        "Published": str(j["published"])
    }
    metadata.append(metadatas)

## Step 14:-
- append the values from the dictionary to the empty dictionary we created outside the for loop in `Step 12`.

In [ ]:
# Done as part of the step 14 as my for loop was breaking.

Step 15:- 
- In step 10, we created a variable called collection which has just the name and the kind of similarity search it will use.
- Now we `add` ids, embeddings, documents and metadata to it.
    - ids -> `Step 11`
    - embeddings -> `Step 7`
    - documents -> `Step 5`
    - metadata -> `Step 12`

- ids — 
    - ChromaDB needs a unique string identifier for every entry so it can find, update, or delete specific chunks later. 
    - FAISS never needed this because FAISS identifies everything by integer row position (0, 1, 2...). 
    - ChromaDB has no concept of row positions — it only knows entries by their string ID.
- embeddings — 
    - The actual vectors. 
    - This is what gets searched when a query comes in. 
    - Same job as adding vectors to a FAISS index with .add().
- documents — 
    - The original text of each chunk. 
    - ChromaDB stores the raw text natively alongside the vector. 
    - FAISS does not do this at all — in the raw FAISS version you had to save a separate metadata_df CSV just to keep the text somewhere. 
    - ChromaDB keeps the text inside itself.
- metadatas — 
    - Extra fields like source, title, url, published. 
    - These are stored as a dictionary per entry so you can retrieve them later alongside the text.

In [23]:
collection.add(
    ids=chunk_ids,
    embeddings=list_embs,
    documents=chunks,
    metadatas=metadata
)